# Healthcare Compliance AI Demo

This notebook demonstrates the existing healthcare compliance AI microservice pipeline step by step using the real project code. It loads sample OCR text, runs deterministic extraction, executes all 19 compliance rules, and shows how to call the running FastAPI service for the main production endpoints.

## 1. Import project services

This section bootstraps the notebook from the project root and imports the same application modules used by the FastAPI service.

In [ ]:
import importlib
import json
import sys
import tempfile
from pathlib import Path

import httpx
import pandas as pd
from docx import Document

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OCRTextInput = importlib.import_module("app.api.schemas.extraction_schema").OCRTextInput
LocalDeterministicAI = importlib.import_module("app.services.ai.local_ai").LocalDeterministicAI
RuleEngine = importlib.import_module("app.services.rule_engine.engine").RuleEngine
rule_registry = importlib.import_module("app.services.rule_engine.registry")
get_all_rules = rule_registry.get_all_rules
get_rule_count = rule_registry.get_rule_count
validate_rule_registry = rule_registry.validate_rule_registry

pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT

## 2. Load sample OCR text

The notebook uses the existing `sample_data/sample_extract_request.json` payload so the demo stays aligned with the current extraction contract. The three document types are `treatment_plan`, `clinical_notes`, and `dla20`.

In [ ]:
sample_extract_request = json.loads(
    (PROJECT_ROOT / "sample_data" / "sample_extract_request.json").read_text(encoding="utf-8")
)

claim_context = {
    "claim_id": sample_extract_request["claim_id"],
    "provider_id": sample_extract_request["provider_id"],
    "patient_id": sample_extract_request["patient_id"],
    "claim_date": sample_extract_request["claim_date"],
}

ocr_results = [OCRTextInput.model_validate(item) for item in sample_extract_request["ocr_results"]]
ocr_text_by_type = {item.document_type: item.raw_text for item in ocr_results}

pd.DataFrame(
    [
        {
            "document_type": document_type,
            "character_count": len(raw_text),
            "preview": raw_text[:120] + ("..." if len(raw_text) > 120 else ""),
        }
        for document_type, raw_text in ocr_text_by_type.items()
    ]
)

## 3. Run LocalDeterministicAI extraction

This uses the production extraction class directly. No extraction logic is reimplemented in the notebook.

In [ ]:
ai_service = LocalDeterministicAI()
extracted_claim_data = ai_service.extract(ocr_results, claim_context=claim_context)
extracted_claim_data

## 4. Display extracted claim data as formatted JSON

The extracted claim data below is the same structured model returned by the extraction layer.

In [ ]:
extracted_claim_json = extracted_claim_data.model_dump(mode="json", exclude_none=True)
print(json.dumps(extracted_claim_json, indent=2))

## 5. Load master data from `sample_data/`

The compliance engine expects a BHS matrix, CPT credential mappings, and optional historical claims. This notebook loads the existing sample JSON files used elsewhere in the project.

In [ ]:
bhs_matrix = json.loads((PROJECT_ROOT / "sample_data" / "sample_bhs_matrix.json").read_text(encoding="utf-8"))
cpt_credentials = json.loads(
    (PROJECT_ROOT / "sample_data" / "sample_cpt_credentials.json").read_text(encoding="utf-8")
)
historical_claims = json.loads(
    (PROJECT_ROOT / "sample_data" / "sample_historical_claims.json").read_text(encoding="utf-8")
)

pd.DataFrame(
    [
        {"dataset": "bhs_matrix", "rows": len(bhs_matrix)},
        {"dataset": "cpt_credentials", "rows": len(cpt_credentials)},
        {"dataset": "historical_claims", "rows": len(historical_claims)},
        {"dataset": "registered_rules", "rows": get_rule_count()},
    ]
)

## 6. Run all 19 compliance rules

This section calls the production `RuleEngine` with `get_all_rules()` and the extracted claim data. The same rule registry and scoring engine used by the API are exercised here.

In [ ]:
rule_engine = RuleEngine(get_all_rules())
validation_output = rule_engine.run(
    extracted_claim_data.model_dump(exclude_none=True),
    {"rows": bhs_matrix},
    {"cpt_credentials": cpt_credentials},
    historical_claims,
)

assert get_rule_count() == 19
assert validate_rule_registry() is True
assert len(validation_output.results) == 19

validation_output

## 7. Display compliance summary

The summary below highlights the compliance score, score band, pass/fail lists, and red flag counts.

In [ ]:
summary = {
    "compliance_score": validation_output.compliance_score,
    "score_band": validation_output.score_band,
    "passed_rules": [result.rule_id for result in validation_output.passed_rules],
    "failed_rules": [result.rule_id for result in validation_output.failed_rules],
    "flag_summary": {
        "high": validation_output.high_red_flags,
        "medium": validation_output.medium_red_flags,
        "low": validation_output.low_red_flags,
    },
}

print(json.dumps(summary, indent=2))

## 8. Display the full rule results table

This table shows the complete rule-by-rule output from the production rule engine, including category, priority, status, message, and rule-specific detail.

In [ ]:
rule_results_df = pd.DataFrame([result.model_dump(mode="json") for result in validation_output.results])
rule_results_df["detail"] = rule_results_df["detail"].apply(lambda value: json.dumps(value, sort_keys=True))
rule_results_df

## 9. Optional: call the running FastAPI service

The cells below show how to call the live API if the FastAPI service is already running on `http://127.0.0.1:8001`. Set `RUN_API_DEMOS = True` to execute the HTTP calls.

The validate demo uses the existing sample JSON payload. The analyze-claim demo generates temporary `.docx` files from the sample OCR text, then posts them to the live API so the end-to-end service behavior can be inspected without modifying production code.

In [ ]:
RUN_API_DEMOS = False
API_BASE_URL = "http://127.0.0.1:8001/api/v1/ai"

sample_validate_request = json.loads(
    (PROJECT_ROOT / "sample_data" / "sample_validate_request.json").read_text(encoding="utf-8")
)

if not RUN_API_DEMOS:
    print("Start the FastAPI service locally and set RUN_API_DEMOS = True to execute the HTTP demo cells.")

In [ ]:
if RUN_API_DEMOS:
    with httpx.Client(timeout=60.0) as client:
        health_response = client.get(f"{API_BASE_URL}/health")
        print("GET /health")
        print(json.dumps(health_response.json(), indent=2))
        print("request_id:", health_response.headers.get("X-Request-ID"))
        print()

        validate_response = client.post(f"{API_BASE_URL}/validate", json=sample_validate_request)
        print("POST /validate")
        print("status:", validate_response.status_code)
        print(json.dumps(validate_response.json(), indent=2)[:1500])
        print()

        with tempfile.TemporaryDirectory() as temp_dir_name:
            temp_dir = Path(temp_dir_name)
            generated_files = {}
            for item in ocr_results:
                document = Document()
                for line in item.raw_text.splitlines():
                    document.add_paragraph(line)
                file_path = temp_dir / f"{item.document_type}.docx"
                document.save(file_path)
                generated_files[item.document_type] = file_path

            analyze_files = [
                (
                    "treatment_plan_files",
                    (
                        generated_files["treatment_plan"].name,
                        generated_files["treatment_plan"].read_bytes(),
                        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
                    ),
                ),
                (
                    "clinical_note_files",
                    (
                        generated_files["clinical_notes"].name,
                        generated_files["clinical_notes"].read_bytes(),
                        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
                    ),
                ),
                (
                    "dla20_files",
                    (
                        generated_files["dla20"].name,
                        generated_files["dla20"].read_bytes(),
                        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
                    ),
                ),
            ]

            analyze_data = {
                "claim_id": sample_extract_request["claim_id"],
                "provider_id": sample_extract_request["provider_id"],
                "patient_id": sample_extract_request["patient_id"],
                "claim_date": sample_extract_request["claim_date"],
                "bhs_matrix_json": json.dumps(bhs_matrix),
                "cpt_credentials_json": json.dumps(cpt_credentials),
                "historical_claims_json": json.dumps(historical_claims),
            }

            analyze_response = client.post(
                f"{API_BASE_URL}/analyze-claim",
                data=analyze_data,
                files=analyze_files,
            )
            print("POST /analyze-claim")
            print("status:", analyze_response.status_code)
            print(json.dumps(analyze_response.json(), indent=2)[:2000])